In [1]:
%matplotlib inline
from simulation import *
from math import *
import numpy as np
from scipy import signal
import visualizer

scala = 193
wavelength = 1 * scala
NA = 0.9999
pitch = 1 * scala
# dx = 0.1
dx = pitch / 41
dbu = 1e-6
size = int(ceil(pitch / dx))
grid_info_2d = grid_info_2d_s.create_bloch_mode_fourier([size, size], wavelength, 0.0, NA, [[-pitch/2, -pitch/2], [pitch/2, pitch/2]])
xsize, ysize = grid_info_2d.shape


TODO : 多 pattern 联合优化

In [ ]:
print(grid_info_2d)

In [3]:
mat_metal = share_material_s(material_s.from_epsilon(1+0j, "Metal"))
mat_air = share_material_s(material_s.from_epsilon(0+0j, "Air"))


In [4]:
enable_spectrum_display = True
enable_image_display = True

def display_spectrum(fft):
    if not enable_spectrum_display:
        return
    arr = np.array(fft).reshape((ysize, xsize))
    visualizer.display_complex_field(arr, "fft complex")


def get_raster(input, is_intensity = False):
    raster = xsize * ysize * np.abs(
        np.fft.fftshift(
            np.fft.ifft2(np.array(input).reshape((ysize, xsize))).astype(np.complex64)
        )
    )
    if is_intensity:
        raster = raster * raster
    return raster.flatten().tolist()


def display_image(fft, is_intensity = False):
    if not enable_image_display:
        return
    im = get_raster(fft, is_intensity)
    visualizer.display_complex_field(np.array(im).reshape((ysize, xsize)), "ifft image")
    return im



In [ ]:
n = pitch * 0.2
layer = layer_s()
layer.background_material = mat_air
layer.shapes = [make_circle_shape_s(n, [0, 0], 0, mat_metal)]
layer.parent = [-1]

fft_circle = raster_layer_fourier_transform_s(layer, grid_info_2d, wavelength)
display_spectrum(fft_circle)
nearfield = display_image(fft_circle)

In [ ]:
layer = layer_s()
layer.background_material = mat_air
layer.shapes = [make_rect_shape_s([pitch / 4, pitch / 2], [0, 0], 0, mat_metal)]
layer.parent = [-1]

fft_rect = raster_layer_fourier_transform_s(layer, grid_info_2d, wavelength)
display_spectrum(fft_rect)
display_image(fft_rect)
print("Gibbs oscillation count=", 2 * (1 + wavelength / (pitch / 4)))

In [ ]:
n = pitch / 6
layer = layer_s()
layer.background_material = mat_air
layer.shapes = [make_poly_shape_s([[-n, -n], [n, -n], [0, n]], [0, 0], 0, mat_metal)]
layer.parent = [-1]

fft_tri = raster_layer_fourier_transform_s(layer, grid_info_2d, wavelength)
display_spectrum(fft_tri)
nearfield = display_image(fft_tri)

In [8]:
def low_pass_filter_corner(spectrum, r):
    """
    对角点对齐（Corner-zero）的频谱进行低通滤波
    :param spectrum: 输入的频谱 (numpy array)
    :param r: 截止频率半径
    :return: 滤波后的频谱
    """
    rows, cols = spectrum.shape
    # 创建坐标网格
    # np.ogrid 可以生成用于广播的坐标向量
    u, v = np.ogrid[:rows, :cols]
    
    # 计算到四个角的距离
    # 由于频谱是周期性的，低频位于 (0,0), (0,W), (H,0), (H,W) 附近
    dist_top_left = u**2 + v**2
    dist_top_right = u**2 + (v - cols)**2
    dist_bottom_left = (u - rows)**2 + v**2
    dist_bottom_right = (u - rows)**2 + (v - cols)**2
    
    # 创建掩码：只要距离任意一个角小于等于 r 即可
    mask = (dist_top_left <= r**2) | \
           (dist_top_right <= r**2) | \
           (dist_bottom_left <= r**2) | \
           (dist_bottom_right <= r**2)
    
    # 应用滤波
    filtered_spectrum = spectrum * mask
    return filtered_spectrum, mask

def calculate_metrics(image, w=1.0):
    """
    计算图像的对比度和 NILS
    :param image: 输入图像 (2D numpy array, 强度值)
    :param w: 特征尺寸 (nominal width)，用于计算 NILS
    :return: contrast, nils_map
    """
    # 确保输入是浮点数，避免整数溢出
    img = image.astype(np.float64)
    
    # 1. 计算对比度 (Michelson Contrast)
    # 适用于周期性或带状图形：(Imax - Imin) / (Imax + Imin)
    i_max = np.max(img)
    i_min = np.min(img)
    
    if i_max + i_min == 0:
        contrast = 0
    else:
        contrast = (i_max - i_min) / (i_max + i_min)
    
    # 2. 计算 NILS
    # 计算图像梯度 (导数)
    # np.gradient 返回 [沿y轴梯度, 沿x轴梯度]
    grad_y, grad_x = np.gradient(img)
    
    # 计算梯度的模长 (或者根据你的需求只取特定方向的梯度)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    
    # 为了计算 Log-Slope，图像强度不能为 0
    # 使用 np.where 避免除以零错误
    nils_map = np.zeros_like(img)
    mask = img > 1e-10
    
    # NILS = w * (dI/dx) / I
    nils_map[mask] = w * grad_mag[mask] / img[mask]
    
    # 通常我们关注图像边缘处的最大 NILS
    max_nils = np.max(nils_map)
    
    return {
        "contrast": contrast,
        "max_nils": max_nils,
        # "nils_map": nils_map
    }

def fourier_resample_2d_perfect(image, scale):
    h, w = image.shape
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    
    # 1. FFT 变换并移至中心
    f_coeff = np.fft.ifft2(np.fft.fftshift(image))
    
    new_f_coeff = np.zeros((new_h, new_w), dtype=complex)
    
    # 3. 计算旧频谱和新频谱的中心索引
    # 对于长度 N，fftshift 后的中心点在 N // 2
    center_h_src, center_w_src = h // 2, w // 2
    center_h_dst, center_w_dst = new_h // 2, new_w // 2
    
    # 4. 计算重合区域的尺寸
    # 取新旧尺寸的最小值，确保不会越界
    overlap_h = min(h, new_h)
    overlap_w = min(w, new_w)
    
    # 5. 计算切片的起始和结束位置
    # 这里的关键是确保低频分量完全对齐
    h_start_src = center_h_src - overlap_h // 2
    w_start_src = center_w_src - overlap_w // 2
    
    h_start_dst = center_h_dst - overlap_h // 2
    w_start_dst = center_w_dst - overlap_w // 2

    # 6. 执行频谱复制
    new_f_coeff[h_start_dst:h_start_dst + overlap_h, 
                w_start_dst:w_start_dst + overlap_w] = \
        f_coeff[h_start_src:h_start_src + overlap_h, 
                w_start_src:w_start_src + overlap_w]
    
    # 7. IFFT 回到时域
    # 缩放因子补偿：因为 IFFT 默认除以 new_h*new_w，而 FFT 没除以 h*w
    # 保持能量一致的公式是：res * (new_h * new_w) / (h * w)
    resampled = np.fft.ifftshift(np.fft.fft2(new_f_coeff))
    return np.real(resampled) * (new_h * (new_w / (h * w)))

def smooth_spectrum_2d(f_coeff_centered, scale):
    """
    通过时域补零让频谱看起来更光滑（频域内插）
    :param f_coeff_centered: 已中心化的频谱 (ndarray, complex)
    :param scale: 频谱的分辨率放大倍数
    :return: 尺寸放大后、视觉更平滑的中心化频谱
    """
    # 1. 转回时域 (空间域)
    # 注意：输入已经是 shift 过的，所以要先 ifftshift
    img_space = np.fft.ifft2(np.fft.ifftshift(f_coeff_centered))
    
    # 2. 计算补零后的新尺寸
    h, w = img_space.shape
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    
    # 3. 在时域进行中心补零 (Zero Padding)
    # 这样可以保持图像的相位中心不变
    pad_h = (new_h - h) // 2
    pad_w = (new_w - w) // 2
    
    # 使用 np.pad 进行对称补零
    # 如果是奇数差值，np.pad 的 ((before, after)) 可以处理不对称
    img_padded = np.pad(img_space, 
                        ((pad_h, new_h - h - pad_h), 
                         (pad_w, new_w - w - pad_w)), 
                        mode='constant', constant_values=0)
    
    # 4. 重新变换回频域
    smoothed_f_coeff = np.fft.fftshift(np.fft.fft2(img_padded))
    
    return smoothed_f_coeff


In [ ]:
import matplotlib.pyplot as plt
np.set_printoptions(threshold=np.inf, linewidth=np.inf, precision=3, suppress=True)
pitch_x = pitch
pitch_y = pitch
# source optimization:
# 1. calculate edge_power_map
# for phase in source_map: 
#    diffraction = pupil_filter(fft((thinmask - DC)*phase))
#    edge_power  = sum(conv(diffraction, diffraction^+)) 
#
# 2. threshold(edge_power_map)
# 3. shape constrain
# 4. optimization coefficients
for im in [fft_rect, fft_circle, fft_tri]:
    for i in range(1, 6, 2):
        rx = pitch_x/wavelength*i
        ry = pitch_y/wavelength*i
        source_half_width = 10
        kernel_shape = [int(1 + 2*n) for n in [rx, ry]]
        kernel = np.ones(shape=kernel_shape)
        intput = np.array(im).reshape((ysize, xsize))
        output, mask =low_pass_filter_corner(intput, int(np.floor(max(rx, ry))*NA * 2))
        shifted_output = np.fft.fftshift(output)

        # 1. 获取 output 的尺寸
        h, w = shifted_output.shape
        kh, kw = kernel_shape

        # 2. 计算中心位置
        center_y, center_x = h // 2, w // 2
        y_start = center_y - kh // 2
        y_end   = y_start + kh
        x_start = center_x - kw // 2
        x_end   = x_start + kw
        
        shifted_output[center_x, center_y] = 0
        cropped_spectrum = shifted_output[x_start:x_end, y_start:y_end]
        # cropped_spectrum = smooth_spectrum_2d(cropped_spectrum, source_half_width/rx)
        # print(np.abs(cropped_spectrum))
        edge_power_map = signal.convolve(kernel, np.abs(cropped_spectrum), mode='full')
        # print(edge_power_map)
        plt.figure(figsize=(6, 6))
        plt.imshow(edge_power_map, cmap='viridis') 
        plt.colorbar(label='Intensity') # 显示颜色刻度尺
        plt.title("edge_power_map")
        plt.show()
        # display_image(output)
        # aerial = display_image(output, is_intensity =True)
        # aerial = np.array(aerial).reshape(*grid_info_2d.tilesize)
        # print(calculate_metrics(aerial))

